# DataFrame to LaTeX Table Converter

This notebook provides a dedicated tool (`LatexTableFormatter`) to format DataFrames into publication-ready LaTeX tables, with specialized support for:
- Concatenating central values and uncertainties into LaTeX `$val \pm unc$` syntax.
- Automatic uncertainty-based rounding (PDG standard: 2 significant figures for uncertainty with aligned central value decimal precision).
- Fixed decimal precision rounding.
- Auto-escaping LaTeX underscores and customizable tabular wrappers (`caption`, `label`, `column_format`, `resizebox`).

In [1]:
import math
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd

## 1. `LatexTableFormatter` Class Definition

In [2]:
class LatexTableFormatter:
    """Utilities for formatting Pandas DataFrames and uncertainty pairs into LaTeX tables."""

    @staticmethod
    def format_pm(
        val: float,
        unc: float,
        precision: Optional[int] = None,
        sig_figs: int = 2,
    ) -> str:
        r"""Formats value and uncertainty as '$val \pm unc$' with appropriate rounding.

        Args:
            val: Central observable value.
            unc: Associated uncertainty (SEM, bootstrap error, standard deviation).
            precision: Fixed decimal places to round. If None, dynamically derives
                decimal places from the uncertainty's significant figures (PDG rule).
            sig_figs: Number of significant digits to retain for uncertainty when
                precision is None (default: 2).

        Returns:
            Formatted LaTeX math string '$val \pm unc$'.
        """
        if not np.isfinite(val) or not np.isfinite(unc):
            return "-"

        if precision is not None:
            return f"${val:.{precision}f} \\pm {unc:.{precision}f}$"

        if unc <= 0:
            return f"${val:.4f} \\pm {unc:.4f}$"

        # Dynamic significant-figures rounding based on uncertainty
        order = math.floor(math.log10(abs(unc)))
        decimals = max(0, sig_figs - 1 - order)

        if round(abs(unc), decimals) >= 10 ** (order + 1):
            decimals = max(0, decimals - 1)

        if decimals == 0:
            return f"${round(val):.0f} \\pm {round(unc):.0f}$"

        return f"${val:.{decimals}f} \\pm {unc:.{decimals}f}$"

    @classmethod
    def concatenate_columns(
        cls,
        df: pd.DataFrame,
        val_col: str,
        unc_col: str,
        new_col: Optional[str] = None,
        precision: Optional[int] = None,
        sig_figs: int = 2,
        drop_original: bool = False,
    ) -> pd.DataFrame:
        r"""Concatenates a value column and uncertainty column into a single LaTeX '$val \pm unc$' column.

        Args:
            df: Source DataFrame.
            val_col: Name of central value column.
            unc_col: Name of uncertainty column.
            new_col: Name of resulting merged column. Defaults to val_col.
            precision: Fixed decimal places (optional).
            sig_figs: Significant digits for uncertainty (default: 2).
            drop_original: Whether to drop val_col and unc_col from returned DataFrame.

        Returns:
            Enriched DataFrame with combined LaTeX column.
        """
        target_col = new_col if new_col is not None else val_col
        res = df.copy()

        combined = [
            cls.format_pm(v, u, precision=precision, sig_figs=sig_figs)
            for v, u in zip(res[val_col], res[unc_col])
        ]
        res[target_col] = combined

        if drop_original:
            cols_to_drop = [c for c in [val_col, unc_col] if c != target_col]
            res = res.drop(columns=cols_to_drop)

        return res

    @classmethod
    def merge_uncertainty_pairs(
        cls,
        df: pd.DataFrame,
        pairs: Dict[str, Tuple[str, str]],
        precision: Optional[int] = None,
        sig_figs: int = 2,
        drop_original: bool = True,
    ) -> pd.DataFrame:
        """Merges multiple value/uncertainty column pairs in batch.

        Args:
            df: Source DataFrame.
            pairs: Dictionary mapping '{NewColumnName}': ('val_column', 'unc_column').
            precision: Optional fixed decimal precision.
            sig_figs: Significant figures for uncertainty rounding (default: 2).
            drop_original: Whether to remove original unmerged columns.

        Returns:
            DataFrame with merged LaTeX columns.
        """
        res = df.copy()
        original_cols_to_drop = set()

        for new_col_name, (val_c, unc_c) in pairs.items():
            if val_c in res.columns and unc_c in res.columns:
                res[new_col_name] = [
                    cls.format_pm(v, u, precision=precision, sig_figs=sig_figs)
                    for v, u in zip(res[val_c], res[unc_c])
                ]
                original_cols_to_drop.update([val_c, unc_c])

        if drop_original:
            cols_to_drop = [c for c in original_cols_to_drop if c not in pairs]
            res = res.drop(columns=cols_to_drop, errors="ignore")

        return res

    @classmethod
    def to_latex(
        cls,
        df: pd.DataFrame,
        caption: Optional[str] = None,
        label: Optional[str] = None,
        column_format: Optional[str] = None,
        escape_underscores: bool = True,
        resize_box: bool = False,
        index: bool = False,
    ) -> str:
        r"""Transforms a DataFrame into a LaTeX table snippet.

        Args:
            df: DataFrame to convert.
            caption: Optional table caption.
            label: Optional table label identifier (e.g. 'tab:results').
            column_format: LaTeX column format string (e.g. 'l c c c').
            escape_underscores: Whether to replace '_' with '\_' in text cells and column headers.
            resize_box: Whether to wrap tabular with '\resizebox{\textwidth}{!}{...}'.
            index: Whether to display DataFrame index.

        Returns:
            LaTeX code block string.
        """
        display_df = df.copy()

        if escape_underscores:
            def _escape_math_aware(text: Any) -> Any:
                if not isinstance(text, str):
                    return text
                parts = text.split("$")
                for i in range(len(parts)):
                    if i % 2 == 0:
                        parts[i] = parts[i].replace("_", r"\_")
                return "$".join(parts)

            # Escape column names and object cells without altering math subscripts
            display_df.columns = [_escape_math_aware(str(c)) for c in display_df.columns]
            for col in display_df.select_dtypes(include=["object", "string"]):
                display_df[col] = display_df[col].apply(_escape_math_aware)

        # Automatic column alignment if none supplied
        if column_format is None:
            col_aligns = ["l" if display_df[c].dtype == "object" else "c" for c in display_df.columns]
            column_format = " ".join(col_aligns)

        tabular_code = display_df.to_latex(
            index=index,
            escape=False,
            column_format=column_format,
        )

        if resize_box:
            tabular_code = f"\\resizebox{{\\textwidth}}{{!}}{{\n{tabular_code.strip()}\n}}"

        if caption or label:
            lines = ["\\begin{table}[htbp]", "\\centering"]
            if caption:
                lines.append(f"\\caption{{{caption}}}")
            if label:
                lines.append(f"\\label{{{label}}}")
            lines.append(tabular_code.strip())
            lines.append("\\end{table}")
            return "\n".join(lines) + "\n"

        return tabular_code

## 2. Top-Level Convenience Functions

These helper functions expose direct access to the formatter without explicitly referencing class methods.

In [3]:
def concatenate_pm_columns(
    df: pd.DataFrame,
    val_col: str,
    unc_col: str,
    new_col: Optional[str] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    drop_original: bool = False,
) -> pd.DataFrame:
    r"""Combines two columns into one formatted as '$val \pm unc$'."""
    return LatexTableFormatter.concatenate_columns(
        df=df,
        val_col=val_col,
        unc_col=unc_col,
        new_col=new_col,
        precision=precision,
        sig_figs=sig_figs,
        drop_original=drop_original,
    )


def dataframe_to_latex(
    df: pd.DataFrame,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    escape_underscores: bool = True,
    resize_box: bool = False,
    index: bool = False,
) -> str:
    """Converts a DataFrame directly into a LaTeX table snippet."""
    return LatexTableFormatter.to_latex(
        df=df,
        caption=caption,
        label=label,
        column_format=column_format,
        escape_underscores=escape_underscores,
        resize_box=resize_box,
        index=index,
    )

## 3. Interactive Examples & Usage

### Example A: Synthetic Distance Metrics & Automatic Rounding (PDG Style)

In [4]:
# Create sample DataFrame with diverse magnitudes
sample_df = pd.DataFrame({
    "Feature": ["trig_L2_calo_et", "trig_L2_calo_eta", "Eratio"],
    "Comparison": ["Model V9 vs Model V10", "Model V9 vs Model V10", "Model V9 vs Model V10"],
    "KS_Distance": [0.008336, 0.006995, 0.233209],
    "KS_Uncertainty": [0.005644, 0.006027, 0.008621],
    "Wasserstein": [16.628878, 0.010359, 0.155185],
    "Wasserstein_Uncertainty": [23.797069, 0.012521, 0.005474],
    "JS_Divergence": [0.000370, 0.000194, 0.048428],
    "JS_Uncertainty": [0.000782, 0.000495, 0.003412],
})

display(sample_df)

,Feature,Comparison,KS_Distance,KS_Uncertainty,Wasserstein,Wasserstein_Uncertainty,JS_Divergence,JS_Uncertainty
0,trig_L2_calo_et,Model V9 vs Model V10,0.008336,0.005644,16.628878,23.797069,0.000370,0.000782
1,trig_L2_calo_eta,Model V9 vs Model V10,0.006995,0.006027,0.010359,0.012521,0.000194,0.000495
2,Eratio,Model V9 vs Model V10,0.233209,0.008621,0.155185,0.005474,0.048428,0.003412


### Example B: Merging Multiple Columns with Auto-Rounding (`sig_figs=2`)

In [5]:
# Merge pairs into unified LaTeX columns
pairs = {
    "KS Distance ($D$)": ("KS_Distance", "KS_Uncertainty"),
    "Wasserstein ($W_1$)": ("Wasserstein", "Wasserstein_Uncertainty"),
    "JS Divergence ($D_{\\text{JS}}$)": ("JS_Divergence", "JS_Uncertainty"),
}

# Automatic PDG rounding
formatted_auto = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=None,
    sig_figs=2,
    drop_original=True,
)

print("=== Preview in Notebook ===")
display(formatted_auto)

print("=== Generated LaTeX Code ===")
latex_code = dataframe_to_latex(
    formatted_auto,
    caption="Statistical Distance Metrics with Automatic Uncertainty Rounding.",
    label="tab:distance_metrics",
)
print(latex_code)

=== Preview in Notebook ===


,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$17 \pm 24$,$0.00037 \pm 0.00078$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.010 \pm 0.013$,$0.00019 \pm 0.00049$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


=== Generated LaTeX Code ===
\begin{table}[htbp]
\centering
\caption{Statistical Distance Metrics with Automatic Uncertainty Rounding.}
\label{tab:distance_metrics}
\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $17 \pm 24$ & $0.00037 \pm 0.00078$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.010 \pm 0.013$ & $0.00019 \pm 0.00049$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}
\end{table}



### Example C: Fixed Decimal Precision Rounding (`precision=4`)

In [6]:
# Merge with fixed precision (4 decimal places)
formatted_fixed = LatexTableFormatter.merge_uncertainty_pairs(
    sample_df,
    pairs=pairs,
    precision=4,
    drop_original=True,
)

display(formatted_fixed)
print(dataframe_to_latex(formatted_fixed))

,Feature,Comparison,KS Distance ($D$),Wasserstein ($W_1$),JS Divergence ($D_{\text{JS}}$)
0,trig_L2_calo_et,Model V9 vs Model V10,$0.0083 \pm 0.0056$,$16.6289 \pm 23.7971$,$0.0004 \pm 0.0008$
1,trig_L2_calo_eta,Model V9 vs Model V10,$0.0070 \pm 0.0060$,$0.0104 \pm 0.0125$,$0.0002 \pm 0.0005$
2,Eratio,Model V9 vs Model V10,$0.2332 \pm 0.0086$,$0.1552 \pm 0.0055$,$0.0484 \pm 0.0034$


\begin{tabular}{l l l l l}
\toprule
Feature & Comparison & KS Distance ($D$) & Wasserstein ($W_1$) & JS Divergence ($D_{\text{JS}}$) \\
\midrule
trig\_L2\_calo\_et & Model V9 vs Model V10 & $0.0083 \pm 0.0056$ & $16.6289 \pm 23.7971$ & $0.0004 \pm 0.0008$ \\
trig\_L2\_calo\_eta & Model V9 vs Model V10 & $0.0070 \pm 0.0060$ & $0.0104 \pm 0.0125$ & $0.0002 \pm 0.0005$ \\
Eratio & Model V9 vs Model V10 & $0.2332 \pm 0.0086$ & $0.1552 \pm 0.0055$ & $0.0484 \pm 0.0034$ \\
\bottomrule
\end{tabular}



### Example D: Applying Directly to Real Validation Tables

Use this snippet to load any distance metric CSV table from your validation results and export it directly to LaTeX.

In [7]:
# Example: Scan for a real distance metrics table
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_files = sorted(list((project_root / "notebooks").glob("SingleRegionValidation_*/Tables/*distance_metrics_*.csv")))

if csv_files:
    target_csv = csv_files[0]
    print(f"Loading: {target_csv.name}")
    real_df = pd.read_csv(target_csv)

    metric_pairs = {
        "KS Distance ($D$)": ("KS Distance (D)", "KS Uncertainty"),
        "Wasserstein ($W_1$)": ("Wasserstein Distance", "Wasserstein Uncertainty"),
        "KL Divergence": ("KL Divergence", "KL Uncertainty"),
        "JS Divergence": ("JS Divergence", "JS Uncertainty"),
    }

    # Select only Feature, Comparison, and metric columns
    keep_cols = ["Feature", "Comparison"] + [c for pair in metric_pairs.values() for c in pair]
    sub_df = real_df[[c for c in keep_cols if c in real_df.columns]]

    exported_df = LatexTableFormatter.merge_uncertainty_pairs(
        sub_df,
        pairs=metric_pairs,
        sig_figs=2,
        drop_original=True,
    )

    display(exported_df.head(10))
    print("\n--- Exported LaTeX ---")
    print(dataframe_to_latex(exported_df.head(10), caption=f"Distance metrics from {target_csv.name}"))
else:
    print("No distance metric CSV files discovered.")

No distance metric CSV files discovered.
